# Initial Setup






---



In [61]:
# import necessary classes and functions
import pandas as pd
import numpy as np
from sklearn.model_selection import cross_val_score
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
!pip install catboost
from sklearn.ensemble import GradientBoostingClassifier
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
from catboost import CatBoostClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import precision_score, recall_score, fbeta_score, make_scorer

# Import the Dataset, Conduct Data Exploration, and Data Preprocessing



---



This dataset is pulled from Kaggle. It contains "Class" as the target variable, which is what genre of music a song is, and 14 features. There is no data dictionary to explain these features, but it appears that the features are descriptive features that Spotify uses for audio analysis. There is another dataset available on Kaggle which better describes these features. You can find that dataset here: https://www.kaggle.com/datasets/solomonameh/spotify-music-dataset.


I did not use the linked dataset as it is less clean and much smaller than the dataset I will be using below. I believe the advantages of using a much larger dataset will be necessary for this multiclass classification modeling project.


The features are each defined as:


*   Popularity: ranges from 1 to 100 and is based on the total number of streams in relation to other songs. A score of 100 indicates a highly popular track.
*   Danceability: ranges from 0.0 to 0.1 and scores how suitable a song is for dancing. A score close to 0.1 indicates a more danceable song.
* Energy: ranges from 0 to 1 and measures how lively, intense, or active the song is. A score closer to 1 indicates a higher energy track.
* Key: a whole number which ranges from 0 to 11. This is the musical key of the song. This is categorical in nature as each number represents a specific musical key.
* Loudness: ranges from -40 to 1.35, which is the loudness of the song in decibels. Higher scores indicate louder tracks.
* Mode: binary variable in which 0 represents a song primarily in minor key, and 1 a song primarily in major key.
* Speechiness: measures the presence of spoken word in a track. Higher values indicate higher presence of spoken word.
* Acousticness: ranges from 0 to 1, with scores closer to 1 indicating a more acoustic track.
* Instrumentalness: how instrumental a song is from 0 to 1. 1 indicates a more instrumental track.
* Liveness: how likely a given song is being performed live. Higher values suggest more audience presence.
* Valence: ranging from 0 to 1, lower valence suggests sad or angry emotion while higher valence suggests happier emotion within a song.
* Tempo: the BPM (beats per minute) of a song
* Duration in Milliseconds: the duration of a song
* Time Signature: indicates the number of beats per bar (or measure) for a song



In [2]:
## import dataset from kaggle

import kagglehub
import os

# download latest version and get training and test csv files
# accessed April 27, 2026
music_classification_data = kagglehub.dataset_download("purumalgi/music-genre-classification")
csv_file_path_train = os.path.join(music_classification_data, "train.csv")
csv_file_path_test = os.path.join(music_classification_data, "test.csv")

# label training and test dataframes
df_train = pd.read_csv(csv_file_path_train)
df_test = pd.read_csv(csv_file_path_test)

100%|██████████| 1.29M/1.29M [00:00<00:00, 46.3MB/s]

Extracting files...


In [3]:
# observing first five rows of training data
display(df_train.head(10))

,Artist Name,Track Name,Popularity,danceability,energy,key,loudness,mode,speechiness,acousticness,instrumentalness,liveness,valence,tempo,duration_in min/ms,time_signature,Class
0,Bruno Mars,That's What I Like (feat. Gucci Mane),60.0,0.854,0.564,1.0,-4.964,1,0.0485,0.017100,NaN,0.0849,0.8990,134.071,234596.000000,4,5
1,Boston,Hitch a Ride,54.0,0.382,0.814,3.0,-7.230,1,0.0406,0.001100,0.004010,0.1010,0.5690,116.454,251733.000000,4,10
2,The Raincoats,No Side to Fall In,35.0,0.434,0.614,6.0,-8.334,1,0.0525,0.486000,0.000196,0.3940,0.7870,147.681,109667.000000,4,6
3,Deno,Lingo (feat. J.I & Chunkz),66.0,0.853,0.597,10.0,-6.528,0,0.0555,0.021200,NaN,0.1220,0.5690,107.033,173968.000000,4,5
4,Red Hot Chili Peppers,Nobody Weird Like Me - Remastered,53.0,0.167,0.975,2.0,-4.279,1,0.2160,0.000169,0.016100,0.1720,0.0918,199.060,229960.000000,4,10
5,The Stooges,Search and Destroy - Iggy Pop Mix,53.0,0.235,0.977,6.0,0.878,1,0.1070,0.003530,0.006040,0.1720,0.2410,152.952,208133.000000,4,6
6,Solomon Burke,None Of Us Are Free,48.0,0.674,0.658,5.0,-9.647,0,0.1040,0.404000,0.000001,0.0981,0.6770,143.292,329387.000000,4,2
7,Randy Travis,On the Other Hand,55.0,0.657,0.415,5.0,-9.915,1,0.0250,0.175000,0.000006,0.1320,0.3470,96.030,3.105783,4,4
8,Professional Murder Music,Slow,29.0,0.431,0.776,10.0,-5.403,1,0.0527,0.000022,0.001300,0.1790,0.3180,120.857,237867.000000,4,8
9,Dudu Aharon,"◊ì◊ï◊ì◊ï, ◊ô◊ê◊ú◊ú◊î ◊ô◊ê◊ú◊ú◊î",14.0,0.716,0.885,1.0,-4.348,0,0.0333,0.061400,NaN,0.2530,0.8330,128.043,164093.000000,4,9


From this, we can determine that Artist Name and Track Name are index values, rather than actual features a model can learn from. We can omit these from the feature data.

Additionally, the column "duration in min/ms" is inconsistent. It appears to switch between units of minutes and milliseconds. As a result, I will not use this column as a feature, given the complexities of adjusting all of the data to one unit. Additionally, I don't think that duration would be a very important feature. Perhaps pop songs usually have a shorter duration, but I'm skeptical that duration has any close ties to determine other genres.

In [4]:
# getting information on data within training set
df_train.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 17996 entries, 0 to 17995
Data columns (total 17 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   Artist Name         17996 non-null  object 
 1   Track Name          17996 non-null  object 
 2   Popularity          17568 non-null  float64
 3   danceability        17996 non-null  float64
 4   energy              17996 non-null  float64
 5   key                 15982 non-null  float64
 6   loudness            17996 non-null  float64
 7   mode                17996 non-null  int64  
 8   speechiness         17996 non-null  float64
 9   acousticness        17996 non-null  float64
 10  instrumentalness    13619 non-null  float64
 11  liveness            17996 non-null  float64
 12  valence             17996 non-null  float64
 13  tempo               17996 non-null  float64
 14  duration_in min/ms  17996 non-null  float64
 15  time_signature      17996 non-null  int64  
 16  Clas

In [5]:
# check for missing data in training and test data
print(df_train.isna().sum())
print(df_test.isna().sum())

Artist Name              0
Track Name               0
Popularity             428
danceability             0
energy                   0
key                   2014
loudness                 0
mode                     0
speechiness              0
acousticness             0
instrumentalness      4377
liveness                 0
valence                  0
tempo                    0
duration_in min/ms       0
time_signature           0
Class                    0
dtype: int64
Artist Name              0
Track Name               0
Popularity             227
danceability             0
energy                   0
key                    808
loudness                 0
mode                     0
speechiness              0
acousticness             0
instrumentalness      1909
liveness                 0
valence                  0
tempo                    0
duration_in min/ms       0
time_signature           0
dtype: int64


We observe that popularity, key, and instrumentalness all contain varying amounts of missing data. XGBoost, LightGBM, and CatBoost all have internal algorithms to handle missing data. However, I'd also like to experiment with sklearn's DecisionTreeClassifier, which does not have such capabilities. In order to do that, we will fill in the missing data with median values from each respective feature.

In [87]:
# define target variable for training set
# define feature datasets for both training and test sets, omitting unnecessary index value columns
y_train = df_train['Class']
X_train = df_train.drop(['Artist Name', 'Track Name', 'Class', 'duration_in min/ms'], axis=1)
X_test = df_test.drop(['Artist Name', 'Track Name'], axis=1)

# define categorical and numerical columns
cat_cols = ['key', 'mode']
num_cols = ['Popularity', 'danceability', 'energy', 'loudness', 'speechiness', 'acousticness', 'instrumentalness', 'liveness', 'valence', 'tempo', 'time_signature']

# create preprocessing objects
preprocessor_scaled_imp_ohe = ColumnTransformer([("num", Pipeline([('imputer', SimpleImputer(strategy='median')), ('scaler', StandardScaler())]), num_cols),
                                                ("cat", OneHotEncoder(sparse_output=False), cat_cols)])
preprocessor_imp_ohe = ColumnTransformer([("num", SimpleImputer(strategy='median'), num_cols),
                                          ("cat", OneHotEncoder(sparse_output=False), cat_cols)])
preprocessor_ohe = ColumnTransformer([("cat", OneHotEncoder(sparse_output=False), cat_cols)], remainder='passthrough')

# adjust output
print(preprocessor_scaled_imp_ohe.set_output(transform='pandas'))
print(preprocessor_imp_ohe.set_output(transform='pandas'))
print(preprocessor_ohe.set_output(transform='pandas'))

ColumnTransformer(transformers=[('num',
                                 Pipeline(steps=[('imputer',
                                                  SimpleImputer(strategy='median')),
                                                 ('scaler', StandardScaler())]),
                                 ['Popularity', 'danceability', 'energy',
                                  'loudness', 'speechiness', 'acousticness',
                                  'instrumentalness', 'liveness', 'valence',
                                  'tempo', 'time_signature']),
                                ('cat', OneHotEncoder(sparse_output=False),
                                 ['key', 'mode'])])
ColumnTransformer(transformers=[('num', SimpleImputer(strategy='median'),
                                 ['Popularity', 'danceability', 'energy',
                                  'loudness', 'speechiness', 'acousticness',
                                  'instrumentalness', 'liveness', 'valence',
            

In [7]:
## confirming that preprocessing object will indeed fill in missing data during later ML Model Pipeline

# using preprocessing object on training feature data and checking for missing data
X_train_scaled_imp_ohe = preprocessor_scaled_imp_ohe.fit_transform(X_train)
print("Missing data in training data after imputation filling:")
print(X_train_scaled_imp_ohe.isnull().sum())

Missing data in training data after imputation filling:
num__Popularity          0
num__danceability        0
num__energy              0
num__loudness            0
num__speechiness         0
num__acousticness        0
num__instrumentalness    0
num__liveness            0
num__valence             0
num__tempo               0
num__time_signature      0
cat__key_1.0             0
cat__key_2.0             0
cat__key_3.0             0
cat__key_4.0             0
cat__key_5.0             0
cat__key_6.0             0
cat__key_7.0             0
cat__key_8.0             0
cat__key_9.0             0
cat__key_10.0            0
cat__key_11.0            0
cat__key_nan             0
cat__mode_0              0
cat__mode_1              0
dtype: int64


In [8]:
# observing categories within target variable
y_train.value_counts()

,count
Class,
10,4949
6,2587
9,2524
8,1854
5,1447
1,1373
2,1272
0,625
7,576


From the submission.csv file for this dataset on Kaggle, we can confirm that "Class" (our target variable) is a dummy variable with 11 different categories. They are:

*   0 = Acoustic Folk Genre
*   1 = Alt Music Genre
* 2 = Blues Genre
* 3 = Bollywood Music Genre
* 4 = Country Music Genre
* 5 = HipHop Music Genre
* 6 = Indie Music Genre
* 7 = Instrumental Music Genre
* 8 = Metal Music Genre
* 9 = Pop Music Genre
* 10 = Rock Music Genre



# Data Modeling



---



# Default Decision Tree

As this is a multiclass classification, I believe a decision tree is a good model to begin with. Decision trees are low bias and can adjust to the complexity of the data. Of course, an unconstrained decision tree will result in severe overfitting, but we can always experiment with hyperparameter tuning later on.

I am using a "weighted" version of scoring as this adjusts for imbalance between the target variable categories and accounts for the multiclass target variable. As seen above, some categories such as rock, indie, and pop have a much higher number of observations than country, bollywood, and instrumental. "Weighted" takes the score for each class and takes the average weighted by the number of true instances for each respective class.

I am using F0.5 as my scoring metric. F0.5 considers both precision and recall, but places slightly heavier emphasis on precision. Using an F-score is helpful for imbalanced datasets. Additionally, since F-score considers both precision and recall, if *either* are low, then the F-score will decrease. These are the reasons I went with an F-score as my scoring metric.

In [9]:
# creating f0.5 scoring metric
f05_scorer = make_scorer(fbeta_score, beta=0.5, average='weighted')

In [10]:
# creating default decision tree pipeline
tree_pipe = Pipeline([("prep", preprocessor_imp_ohe), ("model", DecisionTreeClassifier())])

# fit object to training data
tree_pipe.fit(X_train, y_train)

# perform 5-fold cross-validation with F0.5 scoring
tree_score = cross_val_score(tree_pipe, X_train, y_train, cv=5, scoring=f05_scorer)

# print accuracy scores from training data and 5-fold CV
print("Training Accuracy:", tree_pipe.score(X_train, y_train))
print("5-fold CV Mean F0.5-Score:", tree_score.mean())

Training Accuracy: 0.9049788841964881
5-fold CV Mean F0.5-Score: 0.31450932317519686


There are clear signs of overfitting. The training accuracy is notably higher than the scoring metrics upon performing 5-fold cross-validation. This means that this model does not perform well on unseen data. This is likely because this decision tree is unrestricted, and some hyperparameter tuning and restrictions could help prevent the model from overfitting to the training data.

In [65]:
# creating tuned decision tree pipeline
tree_pipe_01 = Pipeline([("prep", preprocessor_imp_ohe), ("model", DecisionTreeClassifier(max_depth=5))])
tree_pipe_02 = Pipeline([("prep", preprocessor_imp_ohe), ("model", DecisionTreeClassifier(max_depth=10))])
tree_pipe_03 = Pipeline([("prep", preprocessor_imp_ohe), ("model", DecisionTreeClassifier(max_depth=25))])

# fit objects to training data
tree_pipe_01.fit(X_train, y_train)
tree_pipe_02.fit(X_train, y_train)
tree_pipe_03.fit(X_train, y_train)

# perform 5-fold cross-validation with F0.5 scoring
tree_score_01 = cross_val_score(tree_pipe_01, X_train, y_train, cv=5, scoring=f05_scorer)
tree_score_02 = cross_val_score(tree_pipe_02, X_train, y_train, cv=5, scoring=f05_scorer)
tree_score_03 = cross_val_score(tree_pipe_03, X_train, y_train, cv=5, scoring=f05_scorer)

# print accuracy scores from training data and 5-fold CV
print("Training Accuracy:", tree_pipe_01.score(X_train, y_train))
print("Model 01 5-fold CV Mean F0.5-Score:", tree_score_01.mean())

print("Training Accuracy:", tree_pipe_02.score(X_train, y_train))
print("Model 02 5-fold CV Mean F0.5-Score:", tree_score_02.mean())

print("Training Accuracy:", tree_pipe_03.score(X_train, y_train))
print("Model 03 5-fold CV Mean F0.5-Score:", tree_score_03.mean())

Training Accuracy: 0.41192487219382085
Model 01 5-fold CV Mean F0.5-Score: 0.32497813947322685
Training Accuracy: 0.5186152478328517
Model 02 5-fold CV Mean F0.5-Score: 0.38687147370478525
Training Accuracy: 0.9019226494776617
Model 03 5-fold CV Mean F0.5-Score: 0.31333184139208425


Adjusting max_depth, particularly in the first two tuned models, reduced overfitting compared to the unrestricted, default decision tree. However, the 5-fold CV scores are still quite low. Let's try using some production-level ML modeling frameworks to see if we can further improve our model's predictive abilities.

# XGBoost

We'll begin with a simple default XGBoost model. I believe this is a logical model to introduce after a simple decision tree as it introduces gradient boosting. Our model is making a large amount of errors, and so rather than attempting to make as little errors as possible, it's likely that learning from feedback will benefit the model. Additionally, XGBoost is a tree-based ensemble modeling framework, so it flows well to follow the simple decision tree models above.

After obtaining the results from the below default model, we'll complete some hyperparameter tuning to attempt to improve the model's predictive abilities. I'll be using the preprocessor_ohe pipeline as XGBoost, LightGBM, and CatBoost all have internal algorithms to handle missing data.

In [48]:
# creating default XGBoost pipeline
xgb_pipe = Pipeline([("prep", preprocessor_ohe), ("model", XGBClassifier(random_state=22))])

#fit object to training data
xgb_pipe.fit(X_train, y_train)

# perform 5-fold cross-validation with F0.5 scoring
xgb_score = cross_val_score(xgb_pipe, X_train, y_train, cv=5, scoring=f05_scorer)

# print accuracy scores from training data and 5-fold CV
print("Training Accuracy:", xgb_pipe.score(X_train, y_train))
print("5-fold CV Mean F0.5-Score:", xgb_score.mean())

Training Accuracy: 0.8246832629473216
5-fold CV Mean F0.5-Score: 0.43536751101792326


There is already considerable improvement in the F0.5 score compared to the default decision tree. There is still some obvious overfitting occurring. Let's see if some hyperparameter tuning can help.

In [49]:
# creating tuned XGBoost pipelines
xgb_pipe_01 = Pipeline([("prep", preprocessor_ohe),
                        ("model", XGBClassifier(n_estimators=600, max_depth=3, learning_rate=0.1, random_state=22))])
xgb_pipe_02 = Pipeline([("prep", preprocessor_ohe),
                        ("model", XGBClassifier(n_estimators=50, max_depth=3, learning_rate=0.1, random_state=22))])
xgb_pipe_03 = Pipeline([("prep", preprocessor_ohe),
                        ("model", XGBClassifier(max_depth=3, learning_rate=0.1, random_state=22))])

# fitting to training data
xgb_pipe_01.fit(X_train, y_train)
xgb_pipe_02.fit(X_train, y_train)
xgb_pipe_03.fit(X_train, y_train)

# perform 5-fold cross-validation with F0.5 scoring
xgb_score_01 = cross_val_score(xgb_pipe_01, X_train, y_train, cv=5, scoring=f05_scorer)
xgb_score_02 = cross_val_score(xgb_pipe_02, X_train, y_train, cv=5, scoring=f05_scorer)
xgb_score_03 = cross_val_score(xgb_pipe_03, X_train, y_train, cv=5, scoring=f05_scorer)

# print accuracy scores from training data and 5-fold CV
print("Training Accuracy:", xgb_pipe_01.score(X_train, y_train))
print("Model 01 5-fold CV Mean F0.5-Score:", xgb_score_01.mean())

print("Training Accuracy:", xgb_pipe_02.score(X_train, y_train))
print("Model 02 5-fold CV Mean F0.5-Score:", xgb_score_02.mean())

print("Training Accuracy:", xgb_pipe_03.score(X_train, y_train))
print("Model 03 5-fold CV Mean F0.5-Score:", xgb_score_03.mean())

Training Accuracy: 0.6262502778395199
Model 01 5-fold CV Mean F0.5-Score: 0.45571482595044843
Training Accuracy: 0.5007779506557013
Model 02 5-fold CV Mean F0.5-Score: 0.43343295194297954
Training Accuracy: 0.5238941987108247
Model 03 5-fold CV Mean F0.5-Score: 0.44829946541740473


I experimented with several different hyperparameter values. I also experimented with increasing regularization via L1 (reg_alpha) and L2 (reg_lambda). Increases in L1 and L2 regularization did not yield consistent improvements in model quality. The model appeared to improve mainly due to a decrease in max_depth and learning_rate from the default values set for these hyperparameters for XGBoost. I experimented with increasing n_estimators as well. This appeared to improve the Training Accuracy, and sometimes the 5-fold CV score, but from the three models above, I would actually go with the third model.

The third model does not have the highest 5-fold CV score, *but* it *does* have a lower disparity between Training Accuracy and 5-fold CV score. Furthermore, all three of the 5-fold CV scores are pretty similar to each other, and any differences are almost negligible. This was consistent for every hyperparameter adjustment I experimented with. The 5-fold CV scores overall didn't increase much from the default XGBoost model. So, increasing n_estimators only increased the disparity between Training Accuracy and 5-fold CV scores, meaning that it led to more overfitting in this instance. The more restricted third model only has a disparity of about 0.07. The second model also has a disparity of 0.07, but the 5-fold CV Score is lower than the default model. Considering all of these reasons, I would choose the hyperparameter tuning of the third model. Let's see if another modeling framework could improve overall predictive abilities.

# LightGBM

We'll begin with a simple default LightGBM model. Here I chose to implement LightGBM as it introduces leaf-wise growth, compared to the level-wise growth of a regular decision tree and XGBoost. I suspect that leaf-wise growth may benefit the model as I'm sure certain features are more hghly correlated with certain genres. For instance, danceability is likely highly correlated with 9 (Pop Music Genre), and likely very low for 0 (Acoustic Music Genre).

After obtaining the results for the default model, we'll complete some hyperparameter tuning to attempt to improve the model's predictive abilities. Please note, adding "verbosity=-1" is not hyperparameter tuning which affects the model's learning. It simply removes the extra informational messages LightGBM produces. The extra messages distract from the scorings, and so I added "verbosity=-1" to the Pipeline to clean up the output message.

In [50]:
# creating default LightGBM pipeline
lgbm_pipe = Pipeline([("prep", preprocessor_ohe), ("model", LGBMClassifier(random_state=22, verbosity=-1))])

#fit object to training data
lgbm_pipe.fit(X_train, y_train)

# perform 5-fold cross-validation with F0.5 scoring
lgbm_score = cross_val_score(lgbm_pipe, X_train, y_train, cv=5, scoring=f05_scorer)

# print accuracy scores from training data and 5-fold CV
print("Training Accuracy:", lgbm_pipe.score(X_train, y_train))
print("5-fold CV Mean F0.5-Score:", lgbm_score.mean())

Training Accuracy: 0.7668926428095132
5-fold CV Mean F0.5-Score: 0.4475940490759306


The disparity between Training Accuracy and 5-fold CV score is smaller than for the default XGBoost model. Let's see if hyperparameter tuning can further decrease this disparity.

In [51]:
# creating tuned LightGBM pipelines
lgbm_pipe_01 = Pipeline([("prep", preprocessor_ohe),
                      ("model", LGBMClassifier(n_estimators=10, num_leaves=10, random_state=22))])
lgbm_pipe_02 = Pipeline([("prep", preprocessor_ohe),
                      ("model", LGBMClassifier(n_estimators=35, num_leaves=10, random_state=22))])
lgbm_pipe_03 = Pipeline([("prep", preprocessor_ohe),
                      ("model", LGBMClassifier(n_estimators=35, num_leaves=5, random_state=22))])

# fitting to training data
lgbm_pipe_01.fit(X_train, y_train)
lgbm_pipe_02.fit(X_train, y_train)
lgbm_pipe_03.fit(X_train, y_train)

# perform 5-fold cross-validation with F0.5 scoring
lgbm_score_01 = cross_val_score(lgbm_pipe_01, X_train, y_train, cv=5, scoring=f05_scorer)
lgbm_score_02 = cross_val_score(lgbm_pipe_02, X_train, y_train, cv=5, scoring=f05_scorer)
lgbm_score_03 = cross_val_score(lgbm_pipe_03, X_train, y_train, cv=5, scoring=f05_scorer)

# print accuracy scores from training data and 5-fold CV
print("Training Accuracy:", lgbm_pipe_01.score(X_train, y_train))
print("Model 01 5-fold CV Mean F0.5-Score:", lgbm_score_01.mean())

print("Training Accuracy:", lgbm_pipe_02.score(X_train, y_train))
print("Model 02 5-fold CV Mean F0.5-Score:", lgbm_score_02.mean())

print("Training Accuracy:", lgbm_pipe_03.score(X_train, y_train))
print("Model 03 5-fold CV Mean F0.5-Score:", lgbm_score_03.mean())

Training Accuracy: 0.49355412313847524
Model 01 5-fold CV Mean F0.5-Score: 0.4318857446699386
Training Accuracy: 0.5368415203378528
Model 02 5-fold CV Mean F0.5-Score: 0.45374895061912823
Training Accuracy: 0.5007223827517226
Model 03 5-fold CV Mean F0.5-Score: 0.44063212300779186


Again, I experimented with several different hyperparameter adjustments in addition to the three models above. L1 and L2 regularization both had minimal, if any, impact on model quality. I found that decreasing the number of estimators as well as the number of leaves were the two most important hyperparameter adjustments. The first and third models have the lowest disparity at 0.06, but each of those models have a 5-fold CV score which is less than the 5-fold CV score for the default LightGBM model. The second model has a difference of 0.09 between Training Accuracy and 5-fold CV score, and the CV score is higher than the default model.

Considering all of the aforementioned aspects, I prefer the second model. In addition to that, I prefer the 2nd tuned LightGBM model to the 3rd tuned XGBoost model. The disparity between Training Accuracy and 5-fold CV score is 0.02 higher for the 2nd tuned LightGBM model, but that difference is quite small, and the CV F0.5 score for the LightGBM model is higher.

Let's try one more modeling framework before finalizing our model choice for use on test data.

# CatBoost

Like XGBoost and LightGBM, CatBoost can handle missing data. Additionally, CatBoost has its own methods to handle categorical data. I'm curious to see how leaving more of the pre-processing to the model's internal algorithms affects its accuracy. Let's experiment with this framework.

I've generated three tuned CatBoost models below for consideration. Running the default CatBoost model was taking longer than 12 minutes to run, and so I stopped the code from completing there. Any positives that would've been recognized from the default model would be outweighed by the lengthy runtime.

In [55]:
# create CatBoost model object
cat_model = CatBoostClassifier(iterations=200, depth=5, learning_rate=0.1, random_state=22, verbose=0, allow_writing_files=False)

# the categorical column 'key' contained missing data. CatBoost cannot automatically handle missing data within categorical columns
# fill missing data in 'key' column with string value "Missing"
X_train[cat_cols] = X_train[cat_cols].fillna("Missing")

# convert categorical columns to string type for CatBoost
for col in cat_cols:
    X_train[col] = X_train[col].astype(str)

# fit object to training data
cat_model.fit(X_train, y_train, cat_features=cat_cols)

# perform 5-fold cross-validation with F0.5 scoring
cat_score = cross_val_score(cat_model, X_train, y_train, cv=5, scoring=f05_scorer, params={"cat_features": cat_cols})

# print accuracy scores from training data and 5-fold CV
print("Training Accuracy:", cat_model.score(X_train, y_train))
print("5-fold CV Mean F0.5 Score:", cat_score.mean())

Training Accuracy: 0.5294509891086908
5-fold CV Mean F0.5 Score: 0.4566450989167935


In [56]:
# create CatBoost model object
cat_model_01 = CatBoostClassifier(iterations=100, depth=3, learning_rate=0.1, random_state=22, verbose=0, allow_writing_files=False)

# the categorical column 'key' contained missing data. CatBoost cannot automatically handle missing data within categorical columns
# fill missing data in 'key' column with string value "Missing"
X_train[cat_cols] = X_train[cat_cols].fillna("Missing")

# convert categorical columns to string type for CatBoost
for col in cat_cols:
    X_train[col] = X_train[col].astype(str)

# fit object to training data
cat_model_01.fit(X_train, y_train, cat_features=cat_cols)

# perform 5-fold cross-validation with F0.5 scoring
cat_score_01 = cross_val_score(cat_model_01, X_train, y_train, cv=5, scoring=f05_scorer, params={"cat_features": cat_cols})

# print accuracy scores from training data and 5-fold CV
print("Training Accuracy:", cat_model_01.score(X_train, y_train))
print("5-fold CV Mean F0.5 Score:", cat_score_01.mean())

Training Accuracy: 0.4857190486774839
5-fold CV Mean F0.5 Score: 0.43499402932103814


In [57]:
# create CatBoost model object
cat_model_02 = CatBoostClassifier(iterations=150, depth=6, learning_rate=0.1, random_state=22, verbose=0, allow_writing_files=False)

# the categorical column 'key' contained missing data. CatBoost cannot automatically handle missing data within categorical columns
# fill missing data in 'key' column with string value "Missing"
X_train[cat_cols] = X_train[cat_cols].fillna("Missing")

# convert categorical columns to string type for CatBoost
for col in cat_cols:
    X_train[col] = X_train[col].astype(str)

# fit object to training data
cat_model_02.fit(X_train, y_train, cat_features=cat_cols)

# perform 5-fold cross-validation with F0.5 scoring
cat_score_02 = cross_val_score(cat_model_02, X_train, y_train, cv=5, scoring=f05_scorer, params={"cat_features": cat_cols})

# print accuracy scores from training data and 5-fold CV
print("Training Accuracy:", cat_model_02.score(X_train, y_train))
print("5-fold CV Mean F0.5 Score:", cat_score_02.mean())

Training Accuracy: 0.5325627917314959
5-fold CV Mean F0.5 Score: 0.45578529599281437


It looks like CatBoost will be the best fit for our data! Particularly, I'd go with "cat_model" as it has a disparity of 0.7 between Training Accuracy and 5-fold CV score. It also has the highest CV score compared to the 3rd tuned XGBoost and 2nd tuned LightGBM models. It did take a bit longer to run, but not unreasonably long. I think the positives from this tuned model outweigh the slightly longer runtime in this instance. This will be the model I use on the test data.

# Use Chosen Model on Test Data



---



In [74]:
# re-train cat_model on full training data
cat_model.fit(X_train, y_train, cat_features=cat_cols)

# preprocess test data
X_test[cat_cols] = X_test[cat_cols].fillna("Missing")
for col in cat_cols:
    X_test[col] = X_test[col].astype(str)

# predict
predictions = cat_model.predict(X_test)

# transform predictions to pandas dataframe
predictions = pd.DataFrame(predictions, columns=['Class'])

# print first 5 rows of predictions and df_test Artist Name and Track Name columns
display(predictions.head(5))
display(df_test[['Artist Name', 'Track Name']].head(5))

,Class
0,10
1,8
2,9
3,9
4,10


,Artist Name,Track Name
0,David Bowie,Space Oddity - 2015 Remaster
1,Crimson Sun,Essence of Creation
2,P!nk,Raise Your Glass
3,Shawn Mendes,Wonder
4,Backstreet Boys,Helpless When She Smiles - Radio Version


The test dataset does not have any target variable to generate a F0.5 score. However, upon printing out the first 5 rows of predictions, and matching them up with the test data Artist Name and Track Name columns, the predictions look to be fairly accurate. The model is doing its best with the limitations of the target categories themselves. For instance, upon a quick internet search, Helpless When She Smiles - Radio Version by the Backstreet Boys is considered a pop-rock ballad. The model sorts this song into 10 (Rock Music Genre) as it needs to predict one single outcome for each song. I'm curious to evaluate further with songs that I am familiar with.

In [82]:
# display predictions from rows '26', '28', '53', '55', '66', '69', '81', '119', '136', '185', '222', '224', '237', '245', '255'
# '265', '279', '284', '305', '328', '4750', '4818', '5257', '5531'
display(predictions.iloc[['26', '28', '53', '55', '66', '69', '81', '119', '136', '185', '222', '224', '237', '245', '255',
                         '265', '279', '284', '305', '328', '4750', '4818', '5257', '5531']])

,Class
26,10
28,10
53,5
55,5
66,9
69,10
81,6
119,4
136,10
185,10


In [83]:
display(df_test.loc[[26, 28, 53, 55, 66, 69, 81, 119, 136, 185, 222, 224, 237, 245, 255,
                         265, 279, 284, 305, 328, 4750, 4818, 5257, 5531], ['Artist Name', 'Track Name']])

,Artist Name,Track Name
26,Santana,Evil Ways - Live at The Woodstock Music & Art ...
28,Nirvana,Smells Like Teen Spirit
53,The Police,Roxanne - Remastered 2003
55,Public Enemy,Rebel Without A Pause
66,Britney Spears,Circus
69,The Beatles,I Am The Walrus - Remastered 2009
81,Sam Ezeh,Sapphire Alley
119,Nickelback,Rockstar
136,"Florida Georgia Line, Luke Bryan",This Is How We Roll
185,ABBA,Gimme! Gimme! Gimme! (A Man After Midnight) - ...


From this, the inconsistency in the model's predictions is much more obvious. There are clear standouts such as Roxanne by The Police being assigned 5 (HipHop Music Genre) and C.R.E.A.M. by Wu-Tang Clan being assigned 9 (Pop Music Genre). I would have sorted Roxanne into 9 (Pop Music Genre) or 10 (Rock Music Genre) and would've definitely sorted C.R.E.A.M. into 5 (HipHop Music Genre). Also, This Is How We Roll by Florida Georgia Line is certainly a 4 (Country Music Genre) song, but it was assigned 10 (Rock Music Genre). A less egregious error would be sorting Lights by Ellie Goulding into 9 (Pop Music Genre) rather than 6 (Indie Music Genre).

Just from the rows that we've displayed, there are a high amount of 10s being assigned. 10 was by far the highest occurring target variable category in the training data. This goes to show how imbalanced datasets are generally biased towards the majority class. This may also be apart of why This Is How We Roll was assigned 10, when it is clearly a country song. 4 (Country Music Genre), was the lowest occurring target variable category in the training data. To see the spread of predictions, I'll display the value counts below.

In [86]:
# displaying predictions value counts and comparing against y_train value counts
print(predictions.value_counts())
print(y_train.value_counts())

Class
10       3185
9        1158
6         874
8         726
5         653
2         441
7         297
0         187
3         125
4          67
Name: count, dtype: int64
Class
10    4949
6     2587
9     2524
8     1854
5     1447
1     1373
2     1272
0      625
7      576
3      402
4      387
Name: count, dtype: int64


Generally, the spread of the target categories is very similar to the training data. It's worth noting, the model did not assign *any* song a value of 1 (Alt Music Genre). There was a fair amount of alternative music in the training dataset, but perhaps not enough for the model to distinguish it from other genres. Perhaps more features need to be added to the model overall. I would suggest adding even more observation too, and hopefully that would even out the frequency of target variable categories.